# Explainable Credit Underwriting & Default Scoring Engine
### Retail Risk Modeling & Explainable AI (XAI) Analysis | IndusInd Bank Preparation

---

## 1. Executive Summary & Regulatory Context
Traditional credit scorecards often rely on opaque machine learning models that can be difficult for loan officers, regulators, and consumers to understand. Under the Reserve Bank of India (RBI) and Fair Lending compliance mandates, adverse credit actions must be accompanied by explicit, non-discriminatory reasons.

This notebook demonstrates:
1. **Imbalance-Aware XGBoost Classifier** trained on 30,000 retail borrower applications with `scale_pos_weight`.
2. **Zero-Leakage Architecture**: Using solely origination-time features (`Age`, `Income`, `LoanAmount`, `CreditScore`, `DefaultHistory`).
3. **Financial Metric Evaluation**: Evaluating via **ROC-AUC, PR-AUC, Precision, and Recall** rather than misleading raw accuracy.
4. **SHAP TreeExplainer (XAI)**: Feature-level attribution providing the **top 3 adverse risk factors** for every loan decision.

In [ ]:
# Cell 1: Environment Imports and Styling
import os
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report, 
    confusion_matrix, roc_curve, precision_recall_curve
)

sns.set_theme(style="whitegrid", font_scale=1.05)
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["font.sans-serif"] = "Arial"
print("Environment initialized successfully.")

In [ ]:
# Cell 2: Load Model Artifacts & Test Dataset
MODEL_DIR = "../model"
DATA_DIR = "../data"

model = joblib.load(os.path.join(MODEL_DIR, "xgb_model.pkl"))
feature_columns = joblib.load(os.path.join(MODEL_DIR, "feature_columns.pkl"))

with open(os.path.join(MODEL_DIR, "model_metadata.json"), "r") as f:
    metadata = json.load(f)

test_df = pd.read_csv(os.path.join(DATA_DIR, "processed", "test_split.csv"))
X_test = test_df[feature_columns]
y_test = test_df["Default"]

print(f"Loaded model successfully. Trained at: {metadata['trained_at']}")
print(f"Test set size: {len(X_test):,} records")
print(f"Class distribution in test set: Default rate = {y_test.mean()*100:.2f}%")

In [ ]:
# Cell 3: Quantitative Model Evaluation
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_pred_proba >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
pr_auc = average_precision_score(y_test, y_pred_proba)

print("=" * 50)
print(f"ROC-AUC Score:           {roc_auc:.4f}")
print(f"PR-AUC (Avg Precision):  {pr_auc:.4f}")
print("=" * 50)
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=["Non-Default (0)", "Default (1)"]))

In [ ]:
# Cell 4: Evaluation Visualizations (ROC, PR, and Confusion Matrix)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color="#0052cc", lw=2.5, label=f"XGBoost (AUC = {roc_auc:.3f})")
axes[0].plot([0, 1], [0, 1], "--", color="grey")
axes[0].set_title("ROC Curve", fontweight="bold")
axes[0].set_xlabel("False Positive Rate")
axes[0].set_ylabel("True Positive Rate (Recall)")
axes[0].legend(loc="lower right")

# Precision-Recall Curve
prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec, prec, color="#d9381e", lw=2.5, label=f"PR Curve (AUC = {pr_auc:.3f})")
axes[1].set_title("Precision-Recall Curve", fontweight="bold")
axes[1].set_xlabel("Recall")
axes[1].set_ylabel("Precision")
axes[1].legend(loc="lower left")

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[2],
            xticklabels=["Non-Default", "Default"], yticklabels=["Non-Default", "Default"])
axes[2].set_title("Confusion Matrix", fontweight="bold")
axes[2].set_xlabel("Predicted")
axes[2].set_ylabel("Actual")

plt.tight_layout()
plt.show()

In [ ]:
# Cell 5: SHAP Global Feature Importance (TreeExplainer)
explainer = shap.TreeExplainer(model)
sample_x = X_test.sample(min(1000, len(X_test)), random_state=42)
shap_values = explainer.shap_values(sample_x)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_x, show=False)
plt.title("Global SHAP Feature Attribution (Credit Underwriting)", fontsize=14, fontweight="bold", pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# Cell 6: Case Study Comparison (Prime vs Subprime Applicant)
cases = pd.DataFrame([
    {"Age": 45, "Income": 130000.0, "LoanAmount": 120000.0, "CreditScore": 810, "DefaultHistory": 0},
    {"Age": 27, "Income": 38000.0, "LoanAmount": 310000.0, "CreditScore": 540, "DefaultHistory": 2}
])[feature_columns]

probs = model.predict_proba(cases)[:, 1]
shap_case_vals = explainer.shap_values(cases)

labels = ["Prime Applicant (Fast-Track)", "Subprime Applicant (Adverse Action)"]
for i, label in enumerate(labels):
    print("=" * 60)
    print(f"{label} -> Predicted Default Probability: {probs[i]*100:.2f}%")
    print("Feature Attributions (SHAP Log-Odds):")
    for feat, sv, val in sorted(zip(feature_columns, shap_case_vals[i], cases.iloc[i]), key=lambda x: x[1], reverse=True):
        direction = "ADVERSE (Increases Risk)" if sv > 0 else "PROTECTIVE (Decreases Risk)"
        print(f"  * {feat:15s} = {val:10.1f} | SHAP: {sv:+.4f} | {direction}")